# SHG mode survey — general-purpose runner

A general operating notebook for `survey.py` and its post-processing scripts: point it at any
config, launch, watch progress, then explore results (tables, summary, geometry cross-sections,
mode-field images, and the raw per-mode `.npz` field data) — all inline, without leaving the
notebook.

Background and theory: see the **SHG Mode Survey Field Guide** artifact (ask Claude for the link
if you don't have it pinned) and `PLAN.md`. Command-line equivalents: `README.md`. For the
original `loaded`-family smoke-test walkthrough specifically (materials check → geometry
convention check → smoke survey), see `SHG_Survey_Driver.ipynb` instead — this notebook is the
general-purpose one for day-to-day use.

**Before you start:**
- **EMode is single-seat.** Close any EMode session on the other machine first.
- **Kernel:** the project venv (same one `survey.py` runs under from the command line).
- **Don't edit or save this `.ipynb` from outside VS Code while a cell is running** — if the file
  changes on disk (e.g. Claude edits it), run **File: Revert File** before running anything, or
  VS Code's buffer can silently overwrite the external edit on the next save.
- Section **B onward reads whatever `RUN_NAME` is currently set to** — by default that's
  `loaded_sweep1` (already has full results) so you can explore before running anything new.
  Section A launches a *different* run name of your choosing; switch `RUN_NAME` to it once that
  finishes to explore it the same way.


In [ ]:
# --- Setup (run first; rerun after editing any .py in this folder -- it reloads them) ---
import glob
import importlib
import json
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

HERE = os.path.abspath('')
assert os.path.exists(os.path.join(HERE, 'survey.py')), f"open this notebook from shg_mode_survey/ (cwd={HERE})"
sys.path.insert(0, HERE)
import geometry as geo
import survey
import shg_physics as sp
importlib.reload(geo); importlib.reload(survey)
PY = sys.executable


def run_script(*args, tail=4000):
    # Run one of this folder's scripts in a subprocess (keeps their matplotlib 'Agg' backend out
    # of the kernel, and keeps a crashed post-processing script from taking the kernel down).
    r = subprocess.run([PY, '-u', *args], cwd=HERE, capture_output=True, text=True)
    print(r.stdout[-tail:])
    if r.returncode:
        print(r.stderr[-tail:])
        raise RuntimeError(f"{args[0]} exited with {r.returncode}")


print('python:', PY)


## A. Pick a config and launch

`GEOMETRIES` in the config module is a list of `(family, params)` tuples — copy
`survey_config.py` (ridge) or `survey_config_loaded.py` (loaded) as a starting point for a new
one, or point straight at an existing config to re-run/extend it (unchanged geometries are
skipped via the fingerprint checkpoint — see the Field Guide's "Pipeline architecture" section).

Launches **detached**, so closing this notebook or restarting the kernel doesn't kill the run.


In [ ]:
NEW_RUN_NAME = 'my_run'                 # <- change this
NEW_CONFIG = 'survey_config_loaded'      # <- and this (a .py filename in this folder, no extension)


def launch_detached(run_name, config_module):
    out = open(os.path.join(HERE, f'survey_stdout_{run_name}.log'), 'a')
    err = open(os.path.join(HERE, f'survey_stdout_{run_name}.err.log'), 'a')
    flags = subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
    args = [PY, '-u', 'survey.py', run_name, config_module]
    try:   # break out of the kernel's job object, if it has one, so a kernel restart doesn't kill the run
        p = subprocess.Popen(args, cwd=HERE, stdout=out, stderr=err, stdin=subprocess.DEVNULL,
                             creationflags=flags | subprocess.CREATE_BREAKAWAY_FROM_JOB)
    except OSError:
        p = subprocess.Popen(args, cwd=HERE, stdout=out, stderr=err, stdin=subprocess.DEVNULL,
                             creationflags=flags)
    print(f"launched PID {p.pid} -> runs/{run_name}/progress.log")
    return p


# Uncomment to actually launch (left commented so re-running this cell by accident doesn't
# start a second copy of a multi-hour sweep):
# sweep_proc = launch_detached(NEW_RUN_NAME, NEW_CONFIG)


In [ ]:
# Monitor: safe to rerun any time, doesn't touch EMode.
plog = os.path.join(HERE, 'runs', NEW_RUN_NAME, 'progress.log')
print(''.join(open(plog, encoding='utf-8').readlines()[-25:]) if os.path.exists(plog) else 'no progress.log yet')
errlog = os.path.join(HERE, f'survey_stdout_{NEW_RUN_NAME}.err.log')
if os.path.exists(errlog) and os.path.getsize(errlog):
    print('--- stderr (tail) ---\n' + open(errlog, encoding='utf-8', errors='replace').read()[-2000:])


## B. Results tables

`RUN_NAME` below drives every cell from here on. Point it at any finished (or in-progress) run.


In [ ]:
RUN_NAME = 'loaded_sweep1'
RUN_DIR = os.path.join(HERE, 'runs', RUN_NAME)

geoms = pd.read_csv(os.path.join(RUN_DIR, 'geometries.csv'))
display(geoms[['fingerprint', 'family', 'status', 'n_pump_modes', 'n_crossings', 'n_refined', 'seconds', 'error']])

pumps = pd.read_csv(os.path.join(RUN_DIR, 'pump_modes.csv'))
display(pumps[['fingerprint', 'pump_id', 'label', 'sym_class', 'te_fraction', 'n_eff_center',
               'scattering_dB_per_m', 'absorption_dB_per_m', 'A_eff_um2',
               'cerenkov_allowed_any', 'cerenkov_best_angle_deg']])

cross = pd.read_csv(os.path.join(RUN_DIR, 'crossings.csv'))
cols = [c for c in ['pump_label', 'sh_label', 'sh_te_fraction', 'wavelength_sh', 'eta_pct_per_W_cm2',
                    'overlap_shape', 'A_eff_pump_um2', 'A_eff_sh_um2', 'pump_scattering_dB_per_m',
                    'sh_scattering_dB_per_m', 'pump_absorption_dB_per_m', 'sh_absorption_dB_per_m',
                    'L_opt_mm', 'peak_efficiency_pct_per_W', 'status'] if c in cross.columns]
print(f"{len(cross)} crossings, {(cross['status'] == 'refined').sum()} refined")
display(cross.sort_values('eta_pct_per_W_cm2', ascending=False)[cols].head(20))


## C. Summary, geometry cross-sections, and mode-field images

All three are pure post-processing (no EMode) and safe to re-run any time — they overwrite their
own output, never the raw CSVs.


In [ ]:
run_script('summarize_run.py', RUN_NAME)
run_script('plot_geometry.py', RUN_NAME)
run_script('plot_crossing.py', RUN_NAME, '20')

display(Markdown(open(os.path.join(RUN_DIR, 'summary', 'summary.md'), encoding='utf-8').read()))


In [ ]:
# Geometry cross-sections -- one per geometry in the run
for p in sorted(glob.glob(os.path.join(RUN_DIR, 'summary', 'geometry', '*.png'))):
    display(Image(filename=p, width=480))


In [ ]:
# Dispersion diagrams (n_eff(lambda) for pumps + every SH track) -- one per geometry
for p in sorted(glob.glob(os.path.join(RUN_DIR, 'summary', 'dispersion', '*.png'))):
    display(Image(filename=p, width=650))


In [ ]:
# Mode-field images (pump |E|, nonlinear drive, SH field, SH |E|) for the top refined crossings
for p in sorted(glob.glob(os.path.join(RUN_DIR, 'plots', '*.png'))):
    display(Image(filename=p))


## D. Cerenkov / radiated SH (Step 1b): optional

No EMode needed -- runs the driven FDFD solve on each exported pump mode. `ALL_PUMPS=False` only
solves pumps already flagged Cerenkov-allowed (faster); set `True` for every pump regardless.


In [ ]:
ALL_PUMPS = False
run_script('cerenkov_run.py', RUN_NAME, *(['--all'] if ALL_PUMPS else []))
cpath = os.path.join(RUN_DIR, 'cerenkov.csv')
if os.path.exists(cpath):
    display(pd.read_csv(cpath)[['label', 'n_eff', 'wavelength_sh', 'cerenkov_allowed_any', 'kappa_pct_per_W_cm',
                                'to_substrate', 'to_topclad', 'to_left', 'to_right', 'balance', 'seconds']])


## E. Inspect one crossing's raw fields

Every refined crossing's full complex E/H fields are exported to `runs/<run>/exports/*.npz`
(1 W-normalized) — this is the same data `plot_crossing.py` reads to make the images above, one
level more raw: pull out any component, any derived quantity, at full resolution. These exports
are **not** committed to git (one sweep's worth is commonly hundreds of MB) — they live locally
in this folder only. Say so if a specific run's exports are worth handing off separately.


In [ ]:
# Pick a refined crossing by row index into the table from Section B (or just edit export_path
# directly -- it's a column in crossings.csv).
refined = cross[cross['status'] == 'refined'].sort_values('eta_pct_per_W_cm2', ascending=False)
row = refined.iloc[0]
print(row[['pump_label', 'sh_label', 'wavelength_sh', 'eta_pct_per_W_cm2', 'export_path']])

z = np.load(os.path.join(HERE, row['export_path']))
print('\narrays in this export:', z.files)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(np.abs(z['pump_Ey']), origin='lower', extent=[z['x'][0], z['x'][-1], z['y'][0], z['y'][-1]], cmap='magma')
axs[0].set_title(f"pump {row['pump_label']}  |Ey|")
axs[1].imshow(np.abs(z['sh_Ey']), origin='lower', extent=[z['x'][0], z['x'][-1], z['y'][0], z['y'][-1]], cmap='magma')
axs[1].set_title(f"SH {row['sh_label']}  |Ey|")
for ax in axs:
    ax.set_xlabel('x [nm]'); ax.set_ylabel('y [nm]')
fig.tight_layout()
